# 2 — Fresh PCA and LDA baselines
Each run independently rebuilds sources, targets, splits and assemblies. PCA is unsupervised pooled-bin full SVD with rank≥8. LDA is explicitly a supervised classifier, not a dimension-matched embedding. Full/reduced interventions refit all stages.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from neuro.notebook_inputs import configuration_from_environment, inspect_inputs, load_inputs, effective_pair_map
from neuro.pseudopopulation import sample_plan, materialize, support
from neuro.notebook_artifacts import ArtifactStore, FitAccounting, target_heads, write_private_json, verify_artifacts, runtime_identity, validate_kernel_identity
from neuro.notebook_support import workspace, usage_summary, source_budget, checked_assemblies, compact_metrics, plot_confusions

configuration = configuration_from_environment()
settings = configuration['execution']
output = workspace()
def private_warning(message, category, filename, lineno, file=None, line=None):
    # Preserve full provenance privately; do not render host paths in shared figures/output.
    warning_path = output / 'warnings-private.jsonl'
    with warning_path.open('a') as handle:
        handle.write(json.dumps({'message': str(message), 'category': category.__name__, 'filename': filename, 'lineno': lineno}) + '\n')
    warning_path.chmod(0o600)
    print(category.__name__ + ': ' + str(message), file=sys.stderr)
warnings.showwarning = private_warning
store = ArtifactStore(output / 'artifacts')
store.save('configuration', configuration)
store.save('kernel_runtime_before_inputs', validate_kernel_identity(configuration))
print('Input mode:', configuration['mode'], '| exploratory computational assay; no biological IID inference')

## Explicit metadata and target boundary
Authenticate only supplied files, regenerate eligibility/assignments, freeze the contributor embargo and complete-support panel before neural interpretation. The public default is invented data. Missing supplied inputs fail; no demo or historical-result fallback.

In [ ]:
inspection = inspect_inputs(configuration)  # Metadata/hash step only for supplied inputs.
store.save('metadata_selection', inspection)  # Protected provenance; never displayed.
print(json.dumps(inspection.audit, sort_keys=True))
assert inspection.audit['contributor_overlap'] == 0
assert inspection.audit['selected_train'] > 0 and inspection.audit['selected_development'] > 0

In [ ]:
data, neural_audit = load_inputs(inspection)  # Supplied neural path remains explicitly review-gated.
store.save('source_data', data)
store.save('neural_audit', neural_audit)
print(json.dumps(neural_audit, sort_keys=True))
assert np.isfinite(data.values[data.observed]).all()
assert np.isnan(data.values[~data.observed]).all()
assert not data.dependency_audit()['overlapping_contributor_ids']
print('Source × unit × bin shape:', data.values.shape, '| units:', data.value_unit, '| clock units:', data.time_unit)

## Freeze development lineage before interventions
Observed computational seeds 0/1 share development seed 0. Reduced training pools use a nested floor-half subset with fixed subset seed. No singleton fallback.

In [ ]:
from neuro.pca_lda import fit_standardizer, raw_features, fit_pca, fit_lda
from neuro.notebook_support import fit_logistic
observed = effective_pair_map(data, permutation_seed=None)
dev_plan = sample_plan(data, split_id='development', rows_per_condition=settings['development_rows_per_condition'], seed=0, effective_conditions=observed)
reference_development = materialize(data, dev_plan)
store.save('fixed_development', reference_development)
accounting = FitAccounting().start()
results = []

## Visible refit/readout grid
Raw features retain unit-major ordering; PCA features are time-major. Both use training-only pooled-bin normalization. Readouts separately scale training features. Three explicit heads are pair/start/goal; logistic convergence and finiteness are mandatory.

In [ ]:
for seed in settings['seeds']:
    for budget in ('full', 'reduced'):
        selected, effective = source_budget(data, observed, reduced=budget == 'reduced', subset_seed=settings['subset_seed'])
        train, development = checked_assemblies(selected, effective, dev_plan, reference_development, config=settings, seed=seed)
        key = f'observed_seed{seed}_{budget}'
        store.save_bundle(key, train, development)
        print(key, 'source accounting:', json.dumps(usage_summary(selected, train), sort_keys=True))
        standardizer = fit_standardizer(train.values)
        raw_train, raw_dev = raw_features(standardizer.transform(train.values)), raw_features(standardizer.transform(development.values))
        pca = fit_pca(train.values, n_components=settings['pca_components'])
        pca_train, pca_dev = pca.transform(train.values), pca.transform(development.values)
        store.save_encoder(key + '_raw', standardizer, kind='raw')
        store.save_encoder(key + '_pca', pca, kind='pca')
        print(key, 'PCA explained variance:', pca.pca.explained_variance_ratio_.round(4).tolist())
        run_results = []
        for representation, train_x, dev_x in (('raw', raw_train, raw_dev), ('pca', pca_train, pca_dev)):
            for head, y in target_heads(train).items():
                model = fit_logistic(train_x, y, C=settings['logistic_C'], max_iter=settings['logistic_max_iter'])
                row = store.readout(key + '_' + representation + '_' + head, model, encoder=key + '_' + representation,
                                    bundle=key, head=head, train_features=train_x, development_features=dev_x)
                run_results.append(row)
        for head, y in target_heads(train).items():
            model = fit_lda(raw_train, y)
            row = store.readout(key + '_lda_' + head, model, encoder=key + '_raw', bundle=key, head=head,
                                train_features=raw_train, development_features=raw_dev)
            run_results.append(row)
        results.extend(run_results)
        print(json.dumps(compact_metrics(run_results), sort_keys=True))
        fig = plot_confusions(run_results, title=key + ': development counts and row-normalized confusions')
        display(fig); plt.close(fig)
store.save('aggregate_metrics', results)

## Training statistics and descriptive projection
The shown projection is not held-out predictive evidence; the numerical held-out scores and confusions above are the assay.

In [ ]:
print('Last training pooled-bin mean/scale:', standardizer.scaler.mean_.round(3).tolist(), standardizer.scaler.scale_.round(3).tolist())
fig, ax = plt.subplots(figsize=(5, 3)); ax.scatter(pca_dev[:, 0], pca_dev[:, 1], s=6)
ax.set(xlabel='PC1, first cue bin', ylabel='PC2, first cue bin', title='Development projection (descriptive only)')
display(fig); plt.close(fig)
print('Uniform balanced-accuracy references:', {h: 1 / len(np.unique(y)) for h, y in target_heads(development).items()})

## Interpretation limits
These are source-safe conditional pseudopopulations, not simultaneous trials. Matching uses known labels; this is not an unknown-trial prediction pipeline. Bins, synthetic rows and computational seeds are dependent. Place-selected units, provisional marker meaning, inherited preprocessing and historical exposure remain limitations. Contributor separation does not remove long-range serial/session confounds. No biological confidence intervals, p-values, mechanism, dynamics or prospective-path claim follows from these figures. The permutation diagnostic is alignment-breaking, not a biological exchangeability test.

In [ ]:
fit_counts = accounting.stop()
store.save('kernel_runtime_final', runtime_identity())
expected_counts = {'pca': 4, 'logistic': 24, 'lda': 12, 'cebra': 0, 'optimizer_updates': 0}
manifest_hash = store.finish(fit_counts=fit_counts, expected_counts=expected_counts)
replay = verify_artifacts(output / 'artifacts', expected_manifest_hash=manifest_hash)
write_private_json(output / 'completion.json', {'manifest_sha256': manifest_hash, 'fit_counts': fit_counts, 'replay': replay})
print(json.dumps(replay, sort_keys=True))
assert replay['status'] == 'fit-blocked-loaded-model-replay-passed'